# HAIR YOLOE auto-labeling

Upload `hair_colab_runtime.zip` through the Colab Enterprise Files pane, then run each cell in order. Runtime files are deleted when the runtime is deleted, so download `hair_label_results.zip` before deleting the runtime. Generated boxes require human review.

In [ ]:
from pathlib import Path
import shutil
import sys

archives = sorted(Path.cwd().rglob("hair_colab_runtime.zip"))
if len(archives) != 1:
    raise RuntimeError(f"Expected one uploaded hair_colab_runtime.zip, found {len(archives)}")
workspace = Path.cwd() / "hair_colab_workspace"
if workspace.exists():
    raise RuntimeError(f"Workspace already exists: {workspace}")
shutil.unpack_archive(archives[0], workspace)
project = workspace / "hair_colab"
sys.path.insert(0, str(project))
print(f"Project extracted to {project}")

In [ ]:
import colab_runtime
colab_runtime.environment_report()

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=project, check=True)

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=project, check=True)

In [ ]:
subprocess.run([sys.executable, "yoloe_autolabel.py", "--config", "config.yaml", "--validate-only", "--require-enabled-count", "89"], cwd=project, check=True)

In [ ]:
pilot_config = colab_runtime.write_pilot_config(project, max_skus=5, max_images=10)
subprocess.run([sys.executable, "yoloe_autolabel.py", "--config", pilot_config.name, "--require-enabled-count", "89"], cwd=project, check=True)

In [ ]:
from IPython.display import Image, display
preview_paths = sorted((project / "pilot_output" / "raw_predictions" / "previews").glob("*"))
for preview_path in preview_paths[:10]:
    display(Image(filename=str(preview_path), width=900))

## Full run
Run the next cell only after reviewing the pilot previews. It processes all 632 shelf images against all 89 enabled classes.

In [ ]:
subprocess.run([sys.executable, "yoloe_autolabel.py", "--config", "config.yaml", "--require-enabled-count", "89"], cwd=project, check=True)

In [ ]:
results_archive = colab_runtime.package_results(project, Path.cwd() / "hair_label_results.zip")
print(f"Created {results_archive}")
print("Download this ZIP from the Files pane before deleting the runtime.")